# Assemble the Stance Records

| Item | Description |
|---|---|
| Goal | Combine the five shard files of T2.2.3 into two corpus tables: one row per legislator turn, and one row per stance record. The checks here decide whether the production run is usable before anything downstream reads it. |
| Input | `shards/records_00.parquet` to `records_04.parquet` in the T2.2 Interim folder; `member_turns.parquet` from T2.2.1 for the worklist. |
| Output | In the T2.2 Processed folder: `stance_turns.parquet`, one row per turn with its coverage, errors, and merge count; `stance_records.parquet`, one row per record with the speaker, the cited sentence numbers, the topic phrase, and the stance. |
| Variables | • `record_idx`: the record's position within its turn<br>• `sentences`: the cited sentence numbers, which index the sentence list in `member_turns.parquet`<br>• `topic_lint`: the directional word found in the topic phrase, or none |
| Run | First run 2026-08-22, which wrote the stored tables. Rebuilt into a scratch copy on 2026-10-31 and found identical; the outputs shown are from that rebuild. The stored August files stay the inputs of the later steps. |


## Step 1. Import packages


In [1]:
import json
from pathlib import Path

import pandas as pd


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction")
NOTEBOOK_NAME = "T2.2.4 - Production - Assemble the Stance Records"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
SHARD_FOLDER = INTERIM_FOLDER / "shards"
STANCES = ("support", "concern")
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Combine the shards and check the coverage

The five shards partition the stored turn order, so their union must be exactly the non-empty worklist: every turn once, none twice.


In [4]:
turns = pd.concat([pd.read_parquet(SHARD_FOLDER / f"records_{s:02d}.parquet").assign(shard=s) for s in range(5)], ignore_index=True)
work = pd.read_parquet(PROCESSED_FOLDER / "member_turns.parquet", columns=["packageId", "turn_idx", "n_sentences"])
work = work[work["n_sentences"] > 0]

assert not turns.duplicated(["packageId", "turn_idx"]).any(), "turn done twice"
missing = set(zip(work["packageId"], work["turn_idx"])) - set(zip(turns["packageId"], turns["turn_idx"]))
assert not missing, f"{len(missing)} worklist turns missing"
assert len(turns) == len(work), "extra turns beyond the worklist"
print(f"turns: {len(turns):,}, exactly the non-empty worklist, no duplicates")
print(turns.groupby("shard").size().to_string())


turns: 220,160, exactly the non-empty worklist, no duplicates
shard
0    44020
1    44034
2    44046
3    44024
4    44036


## Step 4. Make one row per stance record

Each record keeps its turn key, so record, cited sentences, turn text, and transcript stay one lookup apart.


In [5]:
rows = []
for t in turns.itertuples():
    for i, r in enumerate(json.loads(t.records_json)):
        rows.append({
            "packageId": t.packageId, "turn_idx": t.turn_idx, "year": t.year, "chamber": t.chamber,
            "bioGuideId": t.bioGuideId, "surname": t.surname, "honorific": t.honorific, "is_chair": t.is_chair,
            "record_idx": i, "sentences": r["sentences"], "topic": r["topic"], "stance": r["stance"], "topic_lint": r["topic_lint"],
        })
records = pd.DataFrame(rows)

turns.drop(columns=["records_json"]).to_parquet(PROCESSED_FOLDER / "stance_turns.parquet", compression="zstd", index=False)
records.to_parquet(PROCESSED_FOLDER / "stance_records.parquet", compression="zstd", index=False)
print(f"{len(records):,} stance records -> stance_records.parquet")
print(f"{len(turns):,} turns          -> stance_turns.parquet")


32,385 stance records -> stance_records.parquet
220,160 turns          -> stance_turns.parquet


## Step 5. Check the corpus against the development bands

The bands from T2.2.2: a structural error rate near 3 percent, a few percent of lint flags, both stances present. The per-year counts are the first look at the panel of T2.4.


In [6]:
err_rate = (turns["errors"] != "").mean()
per_turn = turns["n_records"].mean()
stances = records["stance"].value_counts()
lint = records["topic_lint"].notna().mean()
pool = records["topic"].str.casefold().str.strip().nunique()

assert set(records["stance"]) <= set(STANCES)
assert (turns["prompt_version"] == "stance-extraction-0.3-dev").all()

print(f"1) turns with any structural error: {err_rate:.2%} [{'OK' if err_rate < 0.05 else 'OUT OF BAND'}]")
print(f"2) records/turn : {per_turn:.2f} [{'OK' if 0.1 <= per_turn <= 3.0 else 'OUT OF BAND'}]")
print(f"3) stance mix   : {stances.to_dict()} (concern share {stances.get('concern', 0) / len(records):.0%})")
print(f"4) topic lint   : {lint:.2%} flagged, a review queue, not a removal")
print(f"5) topic pool   : {pool:,} unique phrases (case folded)")
print(f"6) empty turns  : {(turns['n_records'] == 0).mean():.0%} returned no record, a valid outcome")
print(f"7) merges       : {turns['n_merged'].sum():,} adjacent duplicates collapsed at extraction time")

err_counts = turns.loc[turns["errors"] != "", "errors"].str.split(";").explode().value_counts()
print("\nerror breakdown:")
print(err_counts.to_string())
print("\nrecords by year:")
byyear = records.groupby("year").size()
print(" ".join(f"{y}:{c:,}" for y, c in byyear.items()))


1) turns with any structural error: 1.87% [OK]
2) records/turn : 0.15 [OK]
3) stance mix   : {'concern': 16236, 'support': 16149} (concern share 50%)
4) topic lint   : 2.59% flagged, a review queue, not a removal
5) topic pool   : 24,798 unique phrases (case folded)
6) empty turns  : 94% returned no record, a valid outcome
7) merges       : 1,815 adjacent duplicates collapsed at extraction time

error breakdown:
errors
indices_not_consecutive    4680
index_out_of_range           21
hit_token_cap                18
json_parse                   17
bad_stance                    9
record_not_object             1

records by year:
1996:8 1999:442 2000:740 2001:1,117 2002:1,259 2003:1,578 2004:821 2005:1,150 2006:1,586 2007:1,851 2008:892 2009:1,249 2010:1,390 2011:1,182 2012:1,268 2013:1,312 2014:1,012 2015:1,468 2016:997 2017:1,913 2018:1,192 2019:1,514 2020:952 2021:1,623 2022:847 2023:1,049 2024:647 2025:1,326
